<a href="https://colab.research.google.com/github/mhowlin-web/TP_RAG_ARCA/blob/main/01_descarga_corpus_arca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP RAG y Agentes

## Asistente para consultas sobre trámites de Monotributo en ARCA

En este proyecto voy a construir un sistema RAG para responder preguntas
sobre trámites relacionados con el Monotributo utilizando información
proveniente de fuentes oficiales de ARCA.

El sistema buscará información relevante dentro de un conjunto de documentos
oficiales y utilizará un modelo de lenguaje para generar respuestas basadas
únicamente en los documentos recuperados.

En este primer notebook construyo el corpus documental que utilizaré en las
siguientes etapas del proyecto.

El proceso que realizo en este notebook es:

1. Definir las fuentes oficiales.
2. Descargar automáticamente las páginas.
3. Extraer el contenido textual.
4. Realizar una limpieza básica.
5. Guardar los documentos en formato JSON.

El corpus JSON funciona como una etapa intermedia del procesamiento.

Posteriormente voy a corregir problemas de encoding, dividir los documentos
en chunks, generar embeddings y almacenar esos embeddings junto con sus
metadatos en Pinecone para realizar el retrieval.

## Instalación de librerías

En esta celda instalo las librerías que necesito para descargar las páginas
web y extraer su contenido.

`requests` me permite realizar solicitudes HTTP.

`BeautifulSoup` me permite analizar el HTML y extraer el texto.

`lxml` se utiliza como parser para procesar el HTML.

In [1]:
!pip install -q requests beautifulsoup4 lxml

## Importación de librerías

En esta celda importo las librerías que voy a utilizar durante el notebook.

También importo `Path` para trabajar de forma más segura con las rutas de
archivos y `datetime` para registrar cuándo descargué cada fuente.

In [2]:
import requests
import json

from bs4 import BeautifulSoup
from datetime import datetime, timezone
from pathlib import Path

## Definición de las carpetas del proyecto

En esta celda creo una carpeta `corpus` para almacenar los archivos
intermedios que genero durante la construcción del corpus.

De esta manera, las siguientes notebooks podrán utilizar una ubicación
conocida para leer el corpus.

Mantengo separado el corpus de las notebooks porque el corpus es un dato
del proyecto y no código.

In [3]:
CARPETA_CORPUS = Path("corpus")

CARPETA_CORPUS.mkdir(
    exist_ok=True
)

print(
    f"Carpeta del corpus: {CARPETA_CORPUS.resolve()}"
)

Carpeta del corpus: /content/corpus


## Definición de las fuentes oficiales

En esta celda defino las páginas oficiales de ARCA que voy a utilizar como
corpus inicial.

Cada fuente tiene un identificador, un título y una URL.

Conservo esta información porque posteriormente voy a utilizarla como
metadatos de los documentos y de los chunks.

Esto me permitirá conocer el origen de la información recuperada por el RAG.

In [4]:
FUENTES_ARCA = [
    {
        "id": "inicio",
        "titulo": "Inicio - Ayuda sobre el Monotributo",
        "url": "https://www.arca.gob.ar/monotributo/ayuda/inicio.asp"
    },
    {
        "id": "clave_fiscal",
        "titulo": "Obtención de Clave Fiscal",
        "url": "https://www.arca.gob.ar/monotributo/ayuda/clave-fiscal.asp"
    },
    {
        "id": "constancias",
        "titulo": "Constancias y credenciales",
        "url": "https://www.arca.gob.ar/monotributo/ayuda/constancias-y-credenciales.asp"
    },
    {
        "id": "facturacion",
        "titulo": "Facturación",
        "url": "https://www.arca.gob.ar/monotributo/ayuda/facturacion.asp"
    },
    {
        "id": "recategorizacion",
        "titulo": "Recategorización",
        "url": "https://ftp.arca.gob.ar/monotributo/ayuda/recategorizacion.asp"
    },
    {
        "id": "baja",
        "titulo": "Baja de monotributo",
        "url": "https://www.arca.gob.ar/monotributo/ayuda/baja.asp"
    },
    {
        "id": "desarrollo_actividad",
        "titulo": "Desarrollo de la actividad",
        "url": "https://www.arca.gob.ar/monotributo/ayuda/desarrollo-de-la-actividad.asp"
    },
    {
        "id": "tutoriales",
        "titulo": "Tutoriales sobre Monotributo",
        "url": "https://www.arca.gob.ar/monotributo/ayuda/tutoriales.asp"
    }
]

## Visualización de las fuentes

En esta celda verifico las fuentes que voy a descargar.

Antes de comenzar la descarga quiero comprobar que las URLs estén
correctamente definidas.

In [5]:
print("FUENTES DEL CORPUS INICIAL")
print("=" * 80)

for i, fuente in enumerate(FUENTES_ARCA, start=1):

    print(f"\n{i}. {fuente['titulo']}")
    print(f"   ID: {fuente['id']}")
    print(f"   URL: {fuente['url']}")

FUENTES DEL CORPUS INICIAL

1. Inicio - Ayuda sobre el Monotributo
   ID: inicio
   URL: https://www.arca.gob.ar/monotributo/ayuda/inicio.asp

2. Obtención de Clave Fiscal
   ID: clave_fiscal
   URL: https://www.arca.gob.ar/monotributo/ayuda/clave-fiscal.asp

3. Constancias y credenciales
   ID: constancias
   URL: https://www.arca.gob.ar/monotributo/ayuda/constancias-y-credenciales.asp

4. Facturación
   ID: facturacion
   URL: https://www.arca.gob.ar/monotributo/ayuda/facturacion.asp

5. Recategorización
   ID: recategorizacion
   URL: https://ftp.arca.gob.ar/monotributo/ayuda/recategorizacion.asp

6. Baja de monotributo
   ID: baja
   URL: https://www.arca.gob.ar/monotributo/ayuda/baja.asp

7. Desarrollo de la actividad
   ID: desarrollo_actividad
   URL: https://www.arca.gob.ar/monotributo/ayuda/desarrollo-de-la-actividad.asp

8. Tutoriales sobre Monotributo
   ID: tutoriales
   URL: https://www.arca.gob.ar/monotributo/ayuda/tutoriales.asp


## Función para descargar una página

En esta celda creo una función para descargar el contenido HTML de una URL.

Utilizo un `User-Agent` para identificar la solicitud como proveniente de
un cliente HTTP.

También utilizo `raise_for_status()` para detectar errores HTTP.

In [6]:
def descargar_pagina(url):

    headers = {
        "User-Agent": (
            "Mozilla/5.0 "
            "(compatible; RAG-ARCA-TP/1.0)"
        )
    }

    respuesta = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    respuesta.raise_for_status()

    return respuesta.text

## Prueba de descarga

En esta celda pruebo la función descargando una de las fuentes.

Primero verifico que puedo acceder correctamente a una página oficial de
ARCA antes de descargar todo el corpus.

In [7]:
html_prueba = descargar_pagina(
    FUENTES_ARCA[0]["url"]
)

print(html_prueba[:1000])

<!DOCTYPE html>
<html lang="es">

    <head>
        <meta charset="utf-8">
        <meta http-equiv="X-UA-Compatible" content="IE=edge">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <!-- Meta para los Buscadores -->
        <title>Inicio - Ayuda sobre el monotributo - Monotributo | ARCA</title>
        <meta name="description" content="Toda la informaciÃ³n sobre cÃ³mo darte de alta y realizar gestiones como monotributista">
        <meta name="keywords" content="monotributo, impositivo, montos, recategorizaciÃ³n, rÃ©gimen,">
        <meta name="author" content="ARCA">
        <meta name="robots" content="Index, Follow">
        <!-- opciones: Index, Follow - NoIndex, Follow - Index, NoFollow - NoIndex, NoFollow -->
        <!-- Facebook , google + -->
        <meta property="og:title" content="Inicio - Ayuda sobre el monotributo - Monotributo | ARCA">
        <meta property="og:type" content="article">
        <meta property="og


## Extracción del texto

En esta celda creo una función para convertir el HTML descargado en texto.

Elimino elementos que no forman parte del contenido textual que me interesa,
como scripts, estilos, iframes y elementos gráficos.

Después extraigo el texto visible y realizo una limpieza básica de espacios.

In [8]:
def extraer_texto(html):

    soup = BeautifulSoup(
        html,
        "lxml"
    )

    for elemento in soup([
        "script",
        "style",
        "noscript",
        "iframe",
        "svg"
    ]):
        elemento.decompose()

    texto = soup.get_text(
        separator="\n"
    )

    lineas = []

    for linea in texto.splitlines():

        linea_limpia = " ".join(
            linea.split()
        )

        if linea_limpia:
            lineas.append(
                linea_limpia
            )

    return "\n".join(lineas)

## Prueba de extracción de texto

En esta celda aplico la función de extracción sobre la página que descargué
anteriormente.

Inspecciono el resultado para comprobar que estoy obteniendo contenido
textual útil.

In [9]:
texto_prueba = extraer_texto(
    html_prueba
)

print(texto_prueba[:5000])

Inicio - Ayuda sobre el monotributo - Monotributo | ARCA
Evitar las herramientas de navegaciÃ³n y pasar al contenido
Monotributo
Menu
Inicio
Ayuda
Inicio
Ayuda sobre el monotributo
Inicio
Ayuda sobre el monotributo
Toda la informaciÃ³n sobre cÃ³mo darte de alta y hacer operaciones como monotributista.
Ingresar con clave fiscal
MenÃº de contenidos
QuÃ© es
INSCRIPCIÃN
Inicio
Clave fiscal
CUIT
Domicilio Fiscal ElectrÃ³nico
Jurisdicciones
Actividades
ALTA DE MONOTRIBUTO
Procedimiento
Tipos de monotributo
ParÃ¡metros
JubilaciÃ³n
Obra social
Monotributo unificado
Constancias y credenciales
DESPUÃS DEL ALTA
Desarrollo de la actividad
FacturaciÃ³n
Pagos
RecategorizaciÃ³n
FINALIZACIÃN DE ACTIVIDADES
Baja
Por cese de actividades
De oficio
ExclusiÃ³n
Renuncia
Pasaje al rÃ©gimen general
Ayuda
Inicio
El primer paso es inscribirse ante ARCA para poder despuÃ©s darse de alta en impuestos y utilizar los servicios con clave fiscal.
Para ello es necesario obtener la
clave fiscal
y la
CUIT
.
Luego, ha

## Descarga y procesamiento de todas las fuentes

En esta celda recorro todas las fuentes definidas anteriormente.

Para cada fuente:

1. Descargo la página.
2. Extraigo el texto.
3. Creo un documento estructurado.
4. Guardo sus metadatos.
5. Registro la fecha de descarga.

Si una fuente produce un error, lo registro y continúo con las demás.

In [10]:
corpus = []

for fuente in FUENTES_ARCA:

    print("=" * 80)
    print(f"Procesando: {fuente['titulo']}")
    print(f"URL: {fuente['url']}")

    try:

        html = descargar_pagina(
            fuente["url"]
        )

        texto = extraer_texto(
            html
        )

        documento = {
            "id": fuente["id"],
            "titulo": fuente["titulo"],
            "organismo": "ARCA",
            "url": fuente["url"],
            "fecha_descarga": datetime.now(
                timezone.utc
            ).isoformat(),
            "texto": texto
        }

        corpus.append(
            documento
        )

        print(
            f"OK - {len(texto)} caracteres extraídos"
        )

    except Exception as error:

        print(
            f"ERROR: {error}"
        )

print("\n" + "=" * 80)
print("PROCESO TERMINADO")
print("=" * 80)

print(
    f"Documentos descargados correctamente: {len(corpus)}"
)

Procesando: Inicio - Ayuda sobre el Monotributo
URL: https://www.arca.gob.ar/monotributo/ayuda/inicio.asp
OK - 1994 caracteres extraídos
Procesando: Obtención de Clave Fiscal
URL: https://www.arca.gob.ar/monotributo/ayuda/clave-fiscal.asp
OK - 2666 caracteres extraídos
Procesando: Constancias y credenciales
URL: https://www.arca.gob.ar/monotributo/ayuda/constancias-y-credenciales.asp
OK - 2626 caracteres extraídos
Procesando: Facturación
URL: https://www.arca.gob.ar/monotributo/ayuda/facturacion.asp
OK - 4305 caracteres extraídos
Procesando: Recategorización
URL: https://ftp.arca.gob.ar/monotributo/ayuda/recategorizacion.asp
OK - 4704 caracteres extraídos
Procesando: Baja de monotributo
URL: https://www.arca.gob.ar/monotributo/ayuda/baja.asp
OK - 2301 caracteres extraídos
Procesando: Desarrollo de la actividad
URL: https://www.arca.gob.ar/monotributo/ayuda/desarrollo-de-la-actividad.asp
OK - 2203 caracteres extraídos
Procesando: Tutoriales sobre Monotributo
URL: https://www.arca.gob.ar

## Resumen del corpus obtenido

En esta celda reviso los documentos que descargué.

Muestro la cantidad de caracteres de cada documento para detectar
rápidamente si alguna página produjo un resultado anormalmente pequeño.

In [11]:
print("RESUMEN DEL CORPUS")
print("=" * 80)

for documento in corpus:

    print(f"\nID: {documento['id']}")
    print(f"Título: {documento['titulo']}")
    print(
        f"Caracteres: {len(documento['texto'])}"
    )
    print(
        f"URL: {documento['url']}"
    )

RESUMEN DEL CORPUS

ID: inicio
Título: Inicio - Ayuda sobre el Monotributo
Caracteres: 1994
URL: https://www.arca.gob.ar/monotributo/ayuda/inicio.asp

ID: clave_fiscal
Título: Obtención de Clave Fiscal
Caracteres: 2666
URL: https://www.arca.gob.ar/monotributo/ayuda/clave-fiscal.asp

ID: constancias
Título: Constancias y credenciales
Caracteres: 2626
URL: https://www.arca.gob.ar/monotributo/ayuda/constancias-y-credenciales.asp

ID: facturacion
Título: Facturación
Caracteres: 4305
URL: https://www.arca.gob.ar/monotributo/ayuda/facturacion.asp

ID: recategorizacion
Título: Recategorización
Caracteres: 4704
URL: https://ftp.arca.gob.ar/monotributo/ayuda/recategorizacion.asp

ID: baja
Título: Baja de monotributo
Caracteres: 2301
URL: https://www.arca.gob.ar/monotributo/ayuda/baja.asp

ID: desarrollo_actividad
Título: Desarrollo de la actividad
Caracteres: 2203
URL: https://www.arca.gob.ar/monotributo/ayuda/desarrollo-de-la-actividad.asp

ID: tutoriales
Título: Tutoriales sobre Monotributo
C

## Visualización de un documento

En esta celda inspecciono manualmente uno de los documentos descargados.

Quiero comprobar la calidad de la extracción antes de utilizar estos
documentos en las etapas posteriores del RAG.

En particular, verifico que haya contenido relacionado con Monotributo y
que no haya solamente elementos de navegación.

In [12]:
documento = corpus[0]

print("=" * 80)
print(documento["titulo"])
print("=" * 80)

print(
    documento["texto"][:10000]
)

Inicio - Ayuda sobre el Monotributo
Inicio - Ayuda sobre el monotributo - Monotributo | ARCA
Evitar las herramientas de navegaciÃ³n y pasar al contenido
Monotributo
Menu
Inicio
Ayuda
Inicio
Ayuda sobre el monotributo
Inicio
Ayuda sobre el monotributo
Toda la informaciÃ³n sobre cÃ³mo darte de alta y hacer operaciones como monotributista.
Ingresar con clave fiscal
MenÃº de contenidos
QuÃ© es
INSCRIPCIÃN
Inicio
Clave fiscal
CUIT
Domicilio Fiscal ElectrÃ³nico
Jurisdicciones
Actividades
ALTA DE MONOTRIBUTO
Procedimiento
Tipos de monotributo
ParÃ¡metros
JubilaciÃ³n
Obra social
Monotributo unificado
Constancias y credenciales
DESPUÃS DEL ALTA
Desarrollo de la actividad
FacturaciÃ³n
Pagos
RecategorizaciÃ³n
FINALIZACIÃN DE ACTIVIDADES
Baja
Por cese de actividades
De oficio
ExclusiÃ³n
Renuncia
Pasaje al rÃ©gimen general
Ayuda
Inicio
El primer paso es inscribirse ante ARCA para poder despuÃ©s darse de alta en impuestos y utilizar los servicios con clave fiscal.
Para ello es necesario obtener l

## Guardado del corpus en formato JSON

En esta celda guardo el corpus en la carpeta `corpus`.

Utilizo UTF-8 y `ensure_ascii=False` para conservar correctamente los
caracteres especiales del español.

Este archivo será el corpus intermedio que utilizaré en la siguiente
notebook para corregir posibles problemas de encoding y continuar con el
preprocesamiento.

In [13]:
ARCHIVO_CORPUS = (
    CARPETA_CORPUS /
    "corpus_arca_monotributo.json"
)

with open(
    ARCHIVO_CORPUS,
    "w",
    encoding="utf-8"
) as archivo:

    json.dump(
        corpus,
        archivo,
        ensure_ascii=False,
        indent=4
    )

print(
    f"Corpus guardado en: {ARCHIVO_CORPUS}"
)

print(
    f"Ruta completa: {ARCHIVO_CORPUS.resolve()}"
)

Corpus guardado en: corpus/corpus_arca_monotributo.json
Ruta completa: /content/corpus/corpus_arca_monotributo.json


## Verificación del archivo guardado

En esta celda vuelvo a cargar el archivo JSON que acabo de generar.

Compruebo que puedo leerlo correctamente y que contiene la misma cantidad
de documentos que el corpus que tengo en memoria.

In [14]:
with open(
    ARCHIVO_CORPUS,
    "r",
    encoding="utf-8"
) as archivo:

    corpus_verificado = json.load(
        archivo
    )

print(
    f"Documentos en memoria: {len(corpus)}"
)

print(
    f"Documentos en JSON: {len(corpus_verificado)}"
)

if len(corpus) == len(corpus_verificado):

    print(
        "OK - El archivo fue guardado correctamente."
    )

else:

    print(
        "ATENCIÓN - Las cantidades no coinciden."
    )

Documentos en memoria: 8
Documentos en JSON: 8
OK - El archivo fue guardado correctamente.


## Conclusión

En este notebook construí el corpus documental inicial para el proyecto
RAG sobre trámites de Monotributo en ARCA.

Descargué información desde fuentes oficiales, extraje su contenido textual
y guardé los documentos junto con sus metadatos.

El corpus quedó almacenado en:

`corpus/corpus_arca_monotributo.json`

En el siguiente notebook voy a analizar y corregir posibles problemas de
encoding antes de realizar el chunking.

Después voy a dividir los documentos en fragmentos, generar embeddings y
almacenarlos en Pinecone para construir el retrieval del sistema RAG.